# Get the data

In [17]:
import pandas as pd

df = pd.read_csv('../../../datasets/drug200.csv')


# Geral analyzes

In [18]:
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,DrugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,DrugY


In [19]:
df.describe()

,Age,Na_to_K
count,200.000000,200.000000
mean,44.315000,16.084485
std,16.544315,7.223956
min,15.000000,6.269000
25%,31.000000,10.445500
50%,45.000000,13.936500
75%,58.000000,19.380000
max,74.000000,38.247000


In [20]:
df.dtypes

Age              int64
Sex             object
BP              object
Cholesterol     object
Na_to_K        float64
Drug            object
dtype: object

In [21]:
object_features =['Sex', "BP", "Cholesterol", "Drug"]
for feature in object_features:
    print(f"Feature: {feature}, values: {df[feature].value_counts()}\n\n")

Feature: Sex, values: Sex
M    104
F     96
Name: count, dtype: int64


Feature: BP, values: BP
HIGH      77
LOW       64
NORMAL    59
Name: count, dtype: int64


Feature: Cholesterol, values: Cholesterol
HIGH      103
NORMAL     97
Name: count, dtype: int64


Feature: Drug, values: Drug
DrugY    91
drugX    54
drugA    23
drugC    16
drugB    16
Name: count, dtype: int64




# Data Cleaning

In [22]:
print(f"Missing values in dataframe: \n{df.isnull().sum()}")

Missing values in dataframe: 
Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
Drug           0
dtype: int64


In [23]:
print(f"Duplicated values: {df.duplicated().sum().sum()}")

Duplicated values: 0


In [24]:
numeric_features = ["Age", 'Na_to_K']

for feature in numeric_features:
    if (df[feature].min() < 0):
        print(f"Feature: {feature}, have negative values")
    else:
        print(f"Feature: {feature} don't have negative values")

Feature: Age don't have negative values
Feature: Na_to_K don't have negative values


# Data Engineering

In [25]:
# Sex -> Binary
# BP -> Ordinal 
#  Cholesterol -> Ordinal (how it's only 2 values we can use as binary too)
# Target -> N_classe


from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder

print(f"Dtypes before ENCODER: {df.dtypes}\n\n\n")

for feature in object_features:
    if feature == "Drug":
        encoder = LabelEncoder()
        df[feature] = encoder.fit_transform(df[feature])
    elif feature == 'BP':
        encoder = OrdinalEncoder()
        df[feature] = encoder.fit_transform(df[[feature]])
    else:
        encoder = OneHotEncoder(sparse_output=False)
        df[feature] = encoder.fit_transform(df[[feature]])

print(f"Dtypes after ENCODER: {df.dtypes}")

Dtypes before ENCODER: Age              int64
Sex             object
BP              object
Cholesterol     object
Na_to_K        float64
Drug            object
dtype: object



Dtypes after ENCODER: Age              int64
Sex            float64
BP             float64
Cholesterol    float64
Na_to_K        float64
Drug             int64
dtype: object


# Data tunning

In [26]:
# How we have low feature DATA TUNNING ISN'T NECESSARY (JUST IF THE MODEL CAN'T BE GOOD)


# Split the data

In [27]:
X = df.drop('Drug', axis = 1)
y = df["Drug"]

print(f"Shapes of X: {X.shape}, Y: {y.shape}")

Shapes of X: (200, 5), Y: (200,)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42)

# Train and metrics

In [29]:
import warnings
warnings.simplefilter("ignore")

In [30]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier

models = [LinearRegression(), DecisionTreeClassifier(), RandomForestClassifier(), AdaBoostClassifier(), XGBClassifier(), LGBMClassifier(), KNeighborsClassifier()]

In [34]:

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, mean_squared_error

for model in models:
    print('----------------------------------------')
    print(f"Model: {model.__class__.__name__}")

    pipe = Pipeline(
        steps=[
            ("Scaler", StandardScaler()),
            ("Model", model)
        ]
    )

    # TRAIN THE MODEL
    pipe.fit(X_train, y_train)

    # See if overfitting (predict the train pipe)
    y_preds_train = pipe.predict(X_train)
    mae = mean_absolute_error(y_train, y_preds_train)
    r2 = r2_score(y_train, y_preds_train)
    rmse = root_mean_squared_error(y_train, y_preds_train)
    mse = mean_squared_error(y_train, y_preds_train)
    print(f"Train Metrics: MAE:{mae:.3f} | R2: {r2*100:.2f}% | RMSE: {rmse:.3f} | MSE: {mse:.3f}")

    # See the metrics (test set)
    y_preds_test = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_preds_test)
    r2 = r2_score(y_test, y_preds_test)
    rmse = root_mean_squared_error(y_test, y_preds_test)
    mse = mean_squared_error(y_test, y_preds_test)
    print(f"Train Metrics: MAE:{mae:.3f} | R2: {r2*100:.2f}% | RMSE: {rmse:.3f} | MSE: {mse:.3f}")

----------------------------------------
Model: LinearRegression
Train Metrics: MAE:0.985 | R2: 56.16% | RMSE: 1.137 | MSE: 1.292
Train Metrics: MAE:0.778 | R2: 69.34% | RMSE: 0.931 | MSE: 0.866
----------------------------------------
Model: DecisionTreeClassifier
Train Metrics: MAE:0.000 | R2: 100.00% | RMSE: 0.000 | MSE: 0.000
Train Metrics: MAE:0.000 | R2: 100.00% | RMSE: 0.000 | MSE: 0.000
----------------------------------------
Model: RandomForestClassifier
Train Metrics: MAE:0.000 | R2: 100.00% | RMSE: 0.000 | MSE: 0.000
Train Metrics: MAE:0.025 | R2: 99.11% | RMSE: 0.158 | MSE: 0.025
----------------------------------------
Model: AdaBoostClassifier
Train Metrics: MAE:0.150 | R2: 94.91% | RMSE: 0.387 | MSE: 0.150
Train Metrics: MAE:0.200 | R2: 92.92% | RMSE: 0.447 | MSE: 0.200
----------------------------------------
Model: XGBClassifier
Train Metrics: MAE:0.000 | R2: 100.00% | RMSE: 0.000 | MSE: 0.000
Train Metrics: MAE:0.025 | R2: 99.11% | RMSE: 0.158 | MSE: 0.025
----------